In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

In [2]:
train_path = '/kaggle/input/datasets/crawford/emnist/emnist-balanced-train.csv'
test_path = '/kaggle/input/datasets/crawford/emnist/emnist-balanced-test.csv'

# Load CSV files
train_df = pd.read_csv(train_path, header=None)
test_df = pd.read_csv(test_path, header=None)

# Separate labels and pixel features
X_train_raw = train_df.iloc[:, 1:].values
y_train = train_df.iloc[:, 0].values
X_test_raw = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values

# Normalize pixel values to [0, 1]
X_train_raw = X_train_raw / 255.0
X_test_raw = X_test_raw / 255.0

# Reshape and fix EMNIST orientation anomaly (transpose width and height axes)
X_train = X_train_raw.reshape(-1, 28, 28, 1).transpose(0, 2, 1, 3)
X_test = X_test_raw.reshape(-1, 28, 28, 1).transpose(0, 2, 1, 3)

num_classes = len(np.unique(y_train))
print(f"Training data shape: {X_train.shape}")
# Target mapping dictionary for your MindBuzz reference:
# 0-9: Digits, 10-35: Uppercase Letters, 36-46: Distinct Lowercase Letters
print(f"Number of target classes detected: {num_classes}")

Training data shape: (112800, 28, 28, 1)
Number of target classes detected: 47


In [3]:
# Model 1: Shallow CNN Baseline
def build_shallow_baseline():
    model = models.Sequential([
        Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name="Shallow_Baseline")
    return model

# Model 2: Deep Regularized CNN
def build_deep_regularized():
    model = models.Sequential([
        Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name="Deep_Regularized")
    return model

# Model 3: Residual CNN (Lightweight skip-connections)
def build_residual_cnn():
    inputs = Input(shape=(28, 28, 1))
    
    # Initial block
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    
    # Residual block structure
    residual = x
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual]) # Skip connection
    x = layers.Activation('relu')(x)
    
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name="Residual_CNN")
    return model

# Initialize the models
models_dictionary = {
    "Shallow Baseline": build_shallow_baseline(),
    "Deep Regularized": build_deep_regularized(),
    "Residual CNN": build_residual_cnn()
}

2026-07-19 21:19:42.489119: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
performance_history = {}
trained_models = {}

EPOCHS = 10  # Increase epochs later for full production convergence
BATCH_SIZE = 128

for name, model in models_dictionary.items():
    print(f"\n========================================\nTraining Model: {name}\n========================================")
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.15,
        verbose=1
    )
    
    performance_history[name] = history
    trained_models[name] = model



Training Model: Shallow Baseline
Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 26s 33ms/step - accuracy: 0.7090 - loss: 1.0099 - val_accuracy: 0.8039 - val_loss: 0.6195
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.8262 - loss: 0.5334 - val_accuracy: 0.8278 - val_loss: 0.5036
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.8499 - loss: 0.4476 - val_accuracy: 0.8395 - val_loss: 0.4646
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 33ms/step - accuracy: 0.8635 - loss: 0.3999 - val_accuracy: 0.8353 - val_loss: 0.4760
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 34ms/step - accuracy: 0.8728 - loss: 0.3650 - val_accuracy: 0.8491 - val_loss: 0.4428
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 33ms/step - accuracy: 0.8810 - loss: 0.3396 - val_accuracy: 0.8544 - val_loss: 0.4275
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.8862 - loss: 0.3176 - val_accuracy: 0.8556 - val_loss: 0.4207
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - 

In [5]:
summary_metrics = []

print("\n========================================\nFINAL EVALUATION RUN\n========================================")

for name, model in trained_models.items():
    print(f"\n--- Calculating metrics for {name} ---")
    
    # Get raw class probabilities and extract highest index predictions
    predictions_prob = model.predict(X_test, verbose=0)
    predictions = np.argmax(predictions_prob, axis=1)
    
    # Calculate performance metrics
    accuracy = accuracy_score(y_test, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average='macro')
    
    summary_metrics.append({
        'Model Name': name,
        'Accuracy': accuracy,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall,
        'F1-Score (Macro)': f1
    })
    
    # Generate and print classification report text
    print(classification_report(y_test, predictions))
    
    # Save a clean Confusion Matrix plot to the output directory
    plt.figure(figsize=(14, 11))
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=False, cmap='Blues', fmt='d')
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('Actual Classes')
    plt.xlabel('Predicted Classes')
    plt.savefig(f'confusion_matrix_{name.lower().replace(" ", "_")}.png')
    plt.close()

# Display the comparative final summary table
summary_df = pd.DataFrame(summary_metrics)
print("\n=====================================================================")
print("                      FINAL ARCHITECTURE METRIC COMPARISON          ")
print("=====================================================================")
print(summary_df.to_string(index=False))
print("=====================================================================")

# Find and log the ultimate winner for MindBuzz production deployment
best_model_row = summary_df.loc[summary_df['Accuracy'].idxmax()]
print(f"\nRecommended Model: {best_model_row['Model Name']} with {best_model_row['Accuracy']*100:.2f}% Test Accuracy.")

# Export the best model file
winning_model_instance = trained_models[best_model_row['Model Name']]
winning_model_instance.save("mindbuzz_best_tracing_model.keras")
print("Saved winning model structure as 'mindbuzz_best_tracing_model.keras'.")


FINAL EVALUATION RUN

--- Calculating metrics for Shallow Baseline ---
              precision    recall  f1-score   support

           0       0.63      0.76      0.69       400
           1       0.57      0.60      0.59       400
           2       0.96      0.73      0.83       400
           3       0.98      0.96      0.97       400
           4       0.92      0.92      0.92       400
           5       0.94      0.80      0.87       400
           6       0.91      0.92      0.92       400
           7       0.95      0.94      0.95       400
           8       0.92      0.88      0.90       400
           9       0.67      0.79      0.73       400
          10       0.90      0.95      0.92       400
          11       0.94      0.91      0.92       400
          12       0.94      0.94      0.94       400
          13       0.89      0.89      0.89       400
          14       0.92      0.97      0.94       400
          15       0.68      0.53      0.60       400
         